In [1]:
folder_path = r"C:\Users\User\OneDrive\A4\Data\videos\AWS"

In [2]:
import os
import pandas as pd
from moviepy.editor import VideoFileClip

#folder_path = r"C:\Users\User\OneDrive\A4\Data\videos\AWS"
output_csv = os.path.join(folder_path, "video_durations.csv")

data = []
total_duration = 0
processed = 0
skipped = 0

for file in os.listdir(folder_path):
    if file.lower().endswith(".mp4"):
        file_path = os.path.join(folder_path, file)
        try:
            clip = VideoFileClip(file_path)
            duration = clip.duration  # seconds
            clip.close()

            # Parse filename to extract title and artist
            # Format: "Title_Artist.mp4"
            filename_without_ext = file.rsplit('.', 1)[0]  # Remove .mp4 extension
            
            if '_' in filename_without_ext:
                parts = filename_without_ext.split('_', 1)
                title = parts[0].strip()    # First portion is title
                artist = parts[1].strip()   # Second portion is artist
            else:
                # If no underscore, put whole name as title and leave artist blank
                title = filename_without_ext
                artist = ""

            data.append({
                "filename": file,
                "title": title,               # First portion
                "artist": artist,             # Second portion
                "duration_seconds": duration,
                "duration_minutes": duration / 60,
                "date": "2026-11-01",
                "type": "L"
            })

            total_duration += duration
            processed += 1

        except Exception as e:
            print(f"Skipping corrupted/unreadable file: {file}")
            skipped += 1

# Create DataFrame
df = pd.DataFrame(data)

# Reorder columns
df = df[["artist", "title", "date", "type","filename",  "duration_seconds", "duration_minutes"]]

# Save to CSV
df.to_csv(output_csv, index=False)

# Convert total duration
hours = int(total_duration // 3600)
minutes = int((total_duration % 3600) // 60)
seconds = int(total_duration % 60)

print(f"Processed files: {processed}")
print(f"Skipped files: {skipped}")
print(f"Total duration: {hours}h {minutes}m {seconds}s")
print(f"CSV saved to: {output_csv}")

# Display first few rows as preview
print("\nPreview of data:")
print(df[["title", "artist", "duration_minutes"]].head(10))

Processed files: 456
Skipped files: 0
Total duration: 32h 18m 22s
CSV saved to: C:\Users\User\OneDrive\A4\Data\videos\AWS\video_durations.csv

Preview of data:
                                     title                       artist  \
0        (Everything I Do) I Do It For You                  Bryan Adams   
1            (I Can't Get No) Satisfaction           Rolling Stones The   
2  (I Can't Help) Falling In Love With You                         UB40   
3         (Sittin' On) The Dock of the Bay                 Otis Redding   
4                                  7 Years                 Lukas Graham   
5                   A Change Is Gonna Come                    Sam Cooke   
6                        A Day in the Life                  Beatles The   
7                     A Horse With No Name                      America   
8                         Against the Wind                    Bob Seger   
9                 All Along the Watchtower  Jimi Hendrix Experience The   

   duration_mi

In [3]:
import pandas as pd
import os

# Path to the CSV file generated by your original script
input_csv = r"C:\Users\User\OneDrive\A4\Data\videos\AWS\video_durations.csv"
output_csv = os.path.join(os.path.dirname(input_csv), "artists_with_multiple_songs.csv")

# Read the CSV file
df = pd.read_csv(input_csv)

# Group by artist and count number of songs
artist_counts = df.groupby('artist').size().reset_index(name='number_of_songs')

# Filter artists with more than one song
#artists_multiple_songs = artist_counts[artist_counts['number_of_songs'] > 1]
artists_multiple_songs = artist_counts.query(
    'number_of_songs > 1 and number_of_songs % 3 != 0'
)

# Sort by number of songs in descending order
artists_multiple_songs = artists_multiple_songs.sort_values('number_of_songs', ascending=False)

# Reset index for clean output
artists_multiple_songs = artists_multiple_songs.reset_index(drop=True)

# Save to CSV
artists_multiple_songs.to_csv(output_csv, index=False)

# Print results
print(f"Artists with more than one song: {len(artists_multiple_songs)}")
print(f"\nResults saved to: {output_csv}")
print("\nPreview:")
print(artists_multiple_songs.to_string(index=False))

# Optional: Show which songs each artist has
print("\n" + "="*50)
print("Detailed breakdown by artist:")
print("="*50)

for artist in artists_multiple_songs['artist']:
    songs = df[df['artist'] == artist]['title'].tolist()
    song_count = len(songs)
    print(f"\n{artist} ({song_count} songs):")
    for i, song in enumerate(songs, 1):
        print(f"  {i}. {song}")

Artists with more than one song: 28

Results saved to: C:\Users\User\OneDrive\A4\Data\videos\AWS\artists_with_multiple_songs.csv

Preview:
            artist  number_of_songs
      Taylor Swift               13
Rolling Stones The               10
              Abba                2
            Avicii                2
  ธงไชย แมคอินไตย์                2
           คาราบาว                2
              UB40                2
                U2                2
       Teresa Teng                2
       Roy Orbison                2
       Ray Charles                2
      RICKY NELSON                2
       Post Malone                2
        Police The                2
    Olivia Rodrigo                2
        Neil Young                2
     Lewis Capaldi                2
    Kelly Clarkson                2
          Jessie J                2
      Jeff Buckley                2
      James Taylor                2
             Heart                2
     Hank Williams               